# Early Experimental Data Preparation: `data.zip` and `dataset.zip`

## Purpose and historical role

This notebook is a cleaned presentation of the **early data-construction pipeline** used at the beginning of the dissertation. The retained code cells are **unchanged from the original executed notebook**; only unrelated exploratory/model-inference cells have been removed and explanatory Markdown has been added.

Two archive representations were produced from the same underlying early Seamless Interaction material:

- **`data.zip`** — participant-centric organisation in which each conversation contains separate participant folders with the corresponding processed MP4, WAV, and JSON files. This representation was used by:
  - `03_isolated_branches/04_participation_branch_development.ipynb`

- **`dataset.zip`** — a derived convenience representation containing horizontally concatenated dyadic videos for the early NORMAL and Wrong Partner experiments. The two participant audio streams are merged into a two-channel stereo track. This representation was used by:
  - `02_direct_baseline/01_direct_monolithic_video_only_baseline.ipynb`

These two archives **do not represent two different source datasets**. They are two representations derived from the same early experimental material. They were kept separately because the thesis was still in its initial exploratory stage and the first downstream experiments were developed around different convenient input layouts. In principle, a single participant-centric source representation together with the corresponding transformation step could have supported both experiments.

`dataset.zip` was therefore created later as a convenience for the monolithic split-screen experiment, whereas `data.zip` preserves the participant-centric organisation used by the early participation study.

### Scope

This notebook documents the **early-stage experimental data preparation only**. It should not be confused with the later controlled data protocol used for the consolidated experiments, where 167 eligible source interactions were divided into temporal-reference, development, and final source-disjoint pools.

### Data flow

```text
Seamless Interaction improvised/test shards
                  |
                  v
       download + preprocessing
                  |
                  v
        participant-centric data/
                  |
        +---------+----------------------+
        |                                |
        v                                v
     data.zip                   construct paired videos
        |                                |
        v                                v
Participation branch          dataset/normal_data_amerged
development                   dataset/anomaly_wrong_partner_amerged
                                         |
                                         v
                                     dataset.zip
                                         |
                                         v
                              Direct monolithic baseline
```

**Historical execution note:** the original shard selection and all retained implementation details are intentionally preserved exactly as they were used during the early experiments rather than being retroactively rewritten.


## 1. Enumerate Seamless Interaction test shards

The public `facebook/seamless-interaction` repository is inspected and the Improvised test shards are identified.


In [4]:
from huggingface_hub import list_repo_files

repo_id = "facebook/seamless-interaction"
files = list_repo_files(repo_id, repo_type="dataset")

print("Number of files:", len(files))
for f in files[:100]:
    print(f)

Number of files: 21970
.gitattributes
LICENSE
README.md
improvised/dev/0000/0000.tar
improvised/dev/0000/0001.tar
improvised/dev/0000/0002.tar
improvised/dev/0000/0003.tar
improvised/dev/0000/0004.tar
improvised/dev/0000/0005.tar
improvised/dev/0000/0006.tar
improvised/dev/0000/0007.tar
improvised/dev/0000/0008.tar
improvised/dev/0000/0009.tar
improvised/dev/0000/0010.tar
improvised/dev/0000/0011.tar
improvised/dev/0000/0012.tar
improvised/dev/0000/0013.tar
improvised/dev/0000/0014.tar
improvised/dev/0000/0015.tar
improvised/dev/0000/0016.tar
improvised/dev/0000/0017.tar
improvised/dev/0000/0018.tar
improvised/dev/0000/0019.tar
improvised/dev/0000/0020.tar
improvised/dev/0000/0021.tar
improvised/dev/0000/0022.tar
improvised/dev/0000/0023.tar
improvised/dev/0000/0024.tar
improvised/dev/0000/0025.tar
improvised/dev/0000/0026.tar
improvised/dev/0000/0027.tar
improvised/dev/0000/0028.tar
improvised/dev/0000/0029.tar
improvised/dev/0000/0030.tar
improvised/dev/0000/0031.tar
improvised/dev/0

In [5]:
test_files = [f for f in files if f.startswith("improvised/test")]

print("Number of test shards:", len(test_files))
print(test_files[:10])

Number of test shards: 83
['improvised/test/0000/0000.tar', 'improvised/test/0000/0001.tar', 'improvised/test/0000/0002.tar', 'improvised/test/0000/0003.tar', 'improvised/test/0000/0004.tar', 'improvised/test/0000/0005.tar', 'improvised/test/0000/0006.tar', 'improvised/test/0000/0007.tar', 'improvised/test/0000/0008.tar', 'improvised/test/0000/0009.tar']


## 2. Download and preprocess the early source material

The selected test shards are downloaded. For each participant stream, the original code:

- retains the first 120 seconds of video;
- crops the upper half of the frame;
- resizes the cropped video to 720 pixels in height;
- retains the participant WAV and JSON files;
- removes temporary video files and cached dataset shards after processing.

The exact historical shard slice used in the original notebook is preserved.


In [ ]:
import tarfile
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download
import os
from pathlib import Path
import shutil

hf_dataset_cache = Path.home() / ".cache/huggingface/hub/datasets--facebook--seamless-interaction"


repo_id = "facebook/seamless-interaction"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

tmp_dir = Path("/tmp/tmp_videos")
tmp_shard_dir = Path("/tmp/tmp_shards")

tmp_dir.mkdir(parents=True, exist_ok=True)
tmp_shard_dir.mkdir(parents=True, exist_ok=True)

for shard in test_files[26:]:
    print(f"\n=== Downloading shard: {shard} ===")

    shard_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=shard
    )

    try:
        with tarfile.open(shard_path) as tar:
            for member in tar.getmembers():
                filename = Path(member.name).name

                # -----------------------------
                # Process videos
                # -----------------------------
                if filename.endswith(".mp4"):
                    print("Processing video:", filename)

                    temp_input = tmp_dir / filename
                    output_video = data_dir / filename

                    try:
                        # Extract raw mp4 temporarily
                        with tar.extractfile(member) as src, open(temp_input, "wb") as dst:
                            dst.write(src.read())

                        # FFmpeg preprocessing:
                        # - keep first 120 sec
                        # - crop upper half
                        # - resize to 720 height
                        cmd = [
                            "ffmpeg",
                            "-y",
                            "-i", str(temp_input),
                            "-t", "120",
                            "-vf", "crop=in_w:in_h/2:0:0,scale=-1:720",
                            "-c:v", "libx264",
                            "-preset", "veryfast",
                            "-crf", "23",
                            "-movflags", "+faststart",
                            str(output_video)
                        ]

                        result = subprocess.run(
                            cmd,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True
                        )

                        if result.returncode != 0:
                            print("FFMPEG FAILED for", filename)
                            print(result.stderr)
                            if output_video.exists():
                                output_video.unlink()
                        else:
                            print("Saved:", output_video)

                    finally:
                        # Delete temporary raw video
                        if temp_input.exists():
                            temp_input.unlink()

                # -----------------------------
                # Keep JSON and WAV only
                # -----------------------------
                elif filename.endswith(".json") or filename.endswith(".wav"):
                    print("Saving metadata/audio:", filename)

                    with tar.extractfile(member) as src, open(data_dir / filename, "wb") as dst:
                        dst.write(src.read())

                # -----------------------------
                # Ignore NPZ for now
                # -----------------------------
                else:
                    continue

    finally:
        # Delete the downloaded shard from HF cache
        if os.path.exists(shard_path):
            os.remove(shard_path)
            print("Deleted cached shard:", shard_path)
        # clear the whole dataset cache (blobs + snapshots)
        if hf_dataset_cache.exists():
            shutil.rmtree(hf_dataset_cache)
            print("Cleared HF dataset cache:", hf_dataset_cache)

## 3. Organise the participant-centric `data/` representation

The downloaded files are reorganised by conversation and participant:

```text
data/
└── conversation_id/
    ├── participant_A/
    │   ├── *.mp4
    │   ├── *.wav
    │   └── *.json
    └── participant_B/
        ├── *.mp4
        ├── *.wav
        └── *.json
```

This participant-centric structure is the basis of `data.zip`.


In [ ]:
from pathlib import Path
import shutil

data_dir = Path("data")

files = list(data_dir.glob("*"))

for f in files:
    name = f.stem
    
    parts = name.split("_")
    
    if len(parts) < 4:
        continue
    
    conversation_id = "_".join(parts[:3])
    participant_id = parts[3]
    
    conv_dir = data_dir / conversation_id
    part_dir = conv_dir / participant_id
    
    part_dir.mkdir(parents=True, exist_ok=True)
    
    new_path = part_dir / f.name
    
    shutil.move(str(f), str(new_path))

print("Dataset reorganized by conversation and participant.")

## 4. Early integrity checks and cleanup

The following original checks identify conversation structure, separate dataset-native silent participant records, verify transcript availability, and remove incomplete single-participant conversations. These steps reflect the exact early workflow used before the first participant-centric and monolithic experiments.


In [11]:
from pathlib import Path
from collections import Counter

data_dir = Path("data")

conversations = [d for d in data_dir.iterdir() if d.is_dir()]

stats = Counter()

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print("Total conversations:", len(conversations))
print()

for k in sorted(stats):
    print(f"{k} participants:", stats[k])

Total conversations: 152

0 participants: 1
1 participants: 119
2 participants: 32


In [12]:
from pathlib import Path
import json
import shutil

data_dir = Path("data")
silent_root = data_dir / "silent"
silent_root.mkdir(exist_ok=True)

conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

moved_count = 0
checked_count = 0
errors = []

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # Κοιτάμε μόνο conversations με 2 participants
    if len(participants) != 2:
        continue

    for participant_dir in participants:
        checked_count += 1

        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            errors.append(f"Expected 1 json in {participant_dir}, found {len(json_files)}")
            continue

        json_path = json_files[0]

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                sample_json = json.load(f)

            transcript = sample_json.get("metadata:transcript", None)

            # silent αν είναι [] ή "" ή None ή γενικά empty
            is_silent = not transcript

            if is_silent:
                target_dir = silent_root / conv.name / participant_dir.name
                target_dir.parent.mkdir(parents=True, exist_ok=True)

                print(f"Moving silent participant: {participant_dir} -> {target_dir}")
                shutil.move(str(participant_dir), str(target_dir))
                moved_count += 1

        except Exception as e:
            errors.append(f"{participant_dir}: {e}")

print("\n=== DONE ===")
print("Participants checked:", checked_count)
print("Silent participants moved:", moved_count)

if errors:
    print("\nErrors:")
    for err in errors:
        print("-", err)

Moving silent participant: data/V01_S0173_I00001232/P1318 -> data/silent/V01_S0173_I00001232/P1318
Moving silent participant: data/V01_S0173_I00001232/P1319 -> data/silent/V01_S0173_I00001232/P1319
Moving silent participant: data/V01_S0307_I00001226/P1632 -> data/silent/V01_S0307_I00001226/P1632
Moving silent participant: data/V01_S0307_I00001226/P1633 -> data/silent/V01_S0307_I00001226/P1633

=== DONE ===
Participants checked: 64
Silent participants moved: 4


In [13]:
from pathlib import Path
from collections import Counter

data_dir = Path("data")

conversations = [d for d in data_dir.iterdir() if d.is_dir()]

stats = Counter()

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print("Total conversations:", len(conversations))
print()

for k in sorted(stats):
    print(f"{k} participants:", stats[k])

Total conversations: 153

0 participants: 3
1 participants: 119
2 participants: 31


In [15]:
from pathlib import Path
import json

data_dir = Path("data")

conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

checked_conversations = 0
empty_transcripts = []

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # μόνο conversations με 2 participants
    if len(participants) != 2:
        continue

    checked_conversations += 1

    for participant_dir in participants:
        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            continue

        json_path = json_files[0]

        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)

        transcript = sample_json.get("metadata:transcript")

        # transcript υπάρχει αλλά είναι άδειο
        if transcript == [] or transcript == "":
            empty_transcripts.append((conv.name, participant_dir.name))

print("Checked conversations:", checked_conversations)
print("Participants with empty transcript:", len(empty_transcripts))
print()

for item in empty_transcripts[:20]:
    print(item)

Checked conversations: 30
Participants with empty transcript: 0



In [28]:
import shutil
from pathlib import Path

data_dir = Path("data")

deleted = 0

for conv in data_dir.iterdir():
    if not conv.is_dir():
        continue

    participants = [p for p in conv.iterdir() if p.is_dir()]

    if len(participants) == 1:
        print("Deleting:", conv)
        shutil.rmtree(conv)
        deleted += 1

print("\nDeleted conversations:", deleted)

Deleting: data/V01_S0173_I00001235
Deleting: data/V03_S1088_I00000125
Deleting: data/V03_S0148_I00000542
Deleting: data/V01_S0173_I00001224
Deleting: data/V01_S0563_I00001232
Deleting: data/V03_S1166_I00000372
Deleting: data/V00_S2037_I00001092
Deleting: data/V00_S2036_I00000713
Deleting: data/V01_S0337_I00001110
Deleting: data/V01_S0563_I00001233
Deleting: data/V00_S2037_I00001088
Deleting: data/V00_S2037_I00001093
Deleting: data/V01_S0340_I00001107
Deleting: data/V00_S2019_I00001102
Deleting: data/V01_S0340_I00001111
Deleting: data/V01_S0338_I00001106
Deleting: data/V03_S0180_I00000129
Deleting: data/V00_S2052_I00000641
Deleting: data/V01_S1545_I00000128
Deleting: data/V03_S0180_I00000578
Deleting: data/V00_S2018_I00001011
Deleting: data/V01_S0563_I00001224
Deleting: data/V03_S1166_I00000543
Deleting: data/V03_S0148_I00000541
Deleting: data/V03_S1166_I00000504
Deleting: data/V00_S2047_I00001049
Deleting: data/V00_S2050_I00001127
Deleting: data/V01_S0340_I00001105
Deleting: data/V01_S

In [29]:
# Check again conversations with 2 participants
from pathlib import Path
import json

data_dir = Path("data")

conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

checked_conversations = 0
empty_transcripts = []

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # μόνο conversations με 2 participants
    if len(participants) != 2:
        continue

    checked_conversations += 1

    for participant_dir in participants:
        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            continue

        json_path = json_files[0]

        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)

        transcript = sample_json.get("metadata:transcript")

        # transcript υπάρχει αλλά είναι άδειο
        if transcript == [] or transcript == "":
            empty_transcripts.append((conv.name, participant_dir.name))

print("Checked conversations:", checked_conversations)
print("Participants with empty transcript:", len(empty_transcripts))
print()

for item in empty_transcripts[:20]:
    print(item)

Checked conversations: 30
Participants with empty transcript: 0



## 5. Construct the early Wrong Partner split-screen representation

Starting from the cleaned participant-centric `data/` structure, the original code constructs one early Wrong Partner example per valid conversation by pairing a participant with a participant from another conversation.

The visual streams are horizontally concatenated. Each participant-specific WAV is first represented as mono and the two streams are then merged into a **two-channel stereo** audio track in the derived video.

The resulting files are written to:

```text
dataset/anomaly_wrong_partner_amerged/
```

This is the early Wrong Partner construction used by the direct monolithic baseline and predates the later systematic matched construction used in the 400-case development protocol.


In [15]:



import json
import random
import subprocess
from pathlib import Path

random.seed(42)

data_dir = Path("data")
output_dir = Path("dataset/anomaly_wrong_partner_amerged")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# helper: transcript non-empty?
# --------------------------------------------------
def has_non_empty_transcript(json_path: Path) -> bool:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)
        transcript = sample_json.get("metadata:transcript", None)
        return transcript not in (None, [], "")
    except Exception:
        return False

# --------------------------------------------------
# Step 1: find valid complete conversations
# --------------------------------------------------
valid_conversations = []

for conv in data_dir.iterdir():
    if not conv.is_dir() or conv.name == "silent":
        continue

    participant_dirs = [p for p in conv.iterdir() if p.is_dir()]

    # μόνο conversations με 2 participants
    if len(participant_dirs) != 2:
        continue

    participant_items = []

    for p_dir in participant_dirs:
        mp4_files = list(p_dir.glob("*.mp4"))
        wav_files = list(p_dir.glob("*.wav"))
        json_files = list(p_dir.glob("*.json"))

        if len(mp4_files) == 1 and len(wav_files) == 1 and len(json_files) == 1:
            if has_non_empty_transcript(json_files[0]):
                participant_items.append({
                    "participant_dir": p_dir,
                    "participant_id": p_dir.name,
                    "video": mp4_files[0],
                    "audio": wav_files[0],
                    "json": json_files[0],
                })

    if len(participant_items) == 2:
        participant_items = sorted(participant_items, key=lambda x: x["participant_id"])
        valid_conversations.append((conv.name, participant_items))

print("Valid complete conversations:", len(valid_conversations))

# --------------------------------------------------
# Step 2: flatten all valid participants
# --------------------------------------------------
all_participants = []

for conv_name, items in valid_conversations:
    for item in items:
        all_participants.append((conv_name, item))

print("All valid participants:", len(all_participants))

# --------------------------------------------------
# Step 3: create one wrong_partner anomaly per conversation
# --------------------------------------------------
created = 0
failed = 0
skipped = 0

for conv_name, items in valid_conversations:
    # διάλεξε deterministic left participant από current conversation
    left_item = random.choice(items)

    # διάλεξε participant από άλλη conversation
    candidates = [x for x in all_participants if x[0] != conv_name]

    if not candidates:
        skipped += 1
        print(f"Skipping {conv_name}: no candidate from another conversation")
        continue

    other_conv_name, right_item = random.choice(candidates)

    left_video = left_item["video"]
    left_audio = left_item["audio"]

    right_video = right_item["video"]
    right_audio = right_item["audio"]

    output_name = (
        f"{conv_name}__WRONG__"
        f"{left_item['participant_id']}__FROM_{conv_name}__"
        f"{right_item['participant_id']}__FROM_{other_conv_name}.mp4"
    )
    output_path = output_dir / output_name

    if output_path.exists():
        print("Skipping existing:", output_path.name)
        skipped += 1
        continue

    print(f"\nCreating wrong_partner anomaly for {conv_name}")
    print("Output      :", output_name)
    print("Left video  :", left_video)
    print("Left audio  :", left_audio)
    print("Right video :", right_video)
    print("Right audio :", right_audio)

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(left_video),
        "-i", str(right_video),
        "-i", str(left_audio),
        "-i", str(right_audio),
        "-filter_complex",
        (
            "[0:v]fps=30,setsar=1,scale=-2:720[v0];"
            "[1:v]fps=30,setsar=1,scale=-2:720[v1];"
            "[v0][v1]hstack=inputs=2[v];"
            "[2:a]aresample=48000,"
            "aformat=sample_fmts=fltp:channel_layouts=mono,"
            "volume=1.0[a0];"
            "[3:a]aresample=48000,"
            "aformat=sample_fmts=fltp:channel_layouts=mono,"
            "volume=1.0[a1];"
            "[a0][a1]amerge=inputs=2[a]"
        ),
        "-map", "[v]",
        "-map", "[a]",
        "-t", "120",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-c:a", "aac",
        "-b:a", "192k",
        "-ac", "2",
        "-shortest",
        str(output_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        failed += 1
        print("FFMPEG FAILED")
        print(result.stderr)
    else:
        created += 1
        print("Saved:", output_path)

print("\n=== DONE ===")
print("Created:", created)
print("Failed :", failed)
print("Skipped:", skipped)

Valid complete conversations: 30
All valid participants: 60

Creating wrong_partner anomaly for V00_S2052_I00000636
Output      : V00_S2052_I00000636__WRONG__P1307A__FROM_V00_S2052_I00000636__P1319__FROM_V03_S0148_I00000135.mp4
Left video  : data/V00_S2052_I00000636/P1307A/V00_S2052_I00000636_P1307A.mp4
Left audio  : data/V00_S2052_I00000636/P1307A/V00_S2052_I00000636_P1307A.wav
Right video : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.mp4
Right audio : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.wav
Saved: dataset/anomaly_wrong_partner_amerged/V00_S2052_I00000636__WRONG__P1307A__FROM_V00_S2052_I00000636__P1319__FROM_V03_S0148_I00000135.mp4

Creating wrong_partner anomaly for V03_S0148_I00000135
Output      : V03_S0148_I00000135__WRONG__P1319__FROM_V03_S0148_I00000135__P1506__FROM_V01_S0223_I00000307.mp4
Left video  : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.mp4
Left audio  : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.wav
Right vi

## 6. Construct the corresponding NORMAL split-screen representation

For each valid dyad, the two genuine conversational partners are horizontally concatenated using the same audiovisual construction pattern. Their participant-specific audio streams are merged into the two-channel stereo track.

The resulting files are written to:

```text
dataset/normal_data_amerged/
```


In [7]:
import json
import subprocess
from pathlib import Path

data_dir = Path("data")
output_dir = Path("dataset/normal_data_amerged")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# helper: transcript non-empty?
# --------------------------------------------------
def has_non_empty_transcript(json_path: Path) -> bool:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)
        transcript = sample_json.get("metadata:transcript", None)
        return transcript not in (None, [], "")
    except Exception:
        return False

# --------------------------------------------------
# Step 1: find valid normal conversations
# --------------------------------------------------
valid_conversations = []

for conv in data_dir.iterdir():
    if not conv.is_dir() or conv.name == "silent":
        continue

    participant_dirs = [p for p in conv.iterdir() if p.is_dir()]

    # μόνο conversations με 2 participants
    if len(participant_dirs) != 2:
        continue

    participant_items = []

    for p_dir in participant_dirs:
        mp4_files = list(p_dir.glob("*.mp4"))
        wav_files = list(p_dir.glob("*.wav"))
        json_files = list(p_dir.glob("*.json"))

        if len(mp4_files) == 1 and len(wav_files) == 1 and len(json_files) == 1:
            if has_non_empty_transcript(json_files[0]):
                participant_items.append({
                    "participant_dir": p_dir,
                    "participant_id": p_dir.name,
                    "video": mp4_files[0],
                    "audio": wav_files[0],
                    "json": json_files[0],
                })

    # κρατάμε μόνο conversations όπου και οι 2 participants είναι valid
    if len(participant_items) == 2:
        valid_conversations.append((conv.name, participant_items))

print("Valid normal conversations:", len(valid_conversations))

# --------------------------------------------------
# Step 2: create one normal video per conversation
# --------------------------------------------------
created = 0
failed = 0
skipped = 0

for conv_name, items in valid_conversations:
    left_item = items[0]
    right_item = items[1]

    left_video = left_item["video"]
    left_audio = left_item["audio"]

    right_video = right_item["video"]
    right_audio = right_item["audio"]

    output_name = f"{conv_name}__NORMAL__{left_item['participant_id']}__{right_item['participant_id']}.mp4"
    output_path = output_dir / output_name

    if output_path.exists():
        print("Skipping existing:", output_path.name)
        skipped += 1
        continue

    print(f"\nCreating normal interaction for {conv_name}")
    print("Left video  :", left_video)
    print("Left audio  :", left_audio)
    print("Right video :", right_video)
    print("Right audio :", right_audio)

    cmd = [
    "ffmpeg",
    "-y",
    "-i", str(left_video),
    "-i", str(right_video),
    "-i", str(left_audio),
    "-i", str(right_audio),
    "-filter_complex",
    (
        "[0:v]fps=30,setsar=1,scale=-2:720[v0];"
        "[1:v]fps=30,setsar=1,scale=-2:720[v1];"
        "[v0][v1]hstack=inputs=2[v];"
        "[2:a]aresample=48000,"
        "aformat=sample_fmts=fltp:channel_layouts=mono,"
        "volume=1.0[a0];"
        "[3:a]aresample=48000,"
        "aformat=sample_fmts=fltp:channel_layouts=mono,"
        "volume=1.0[a1];"
        "[a0][a1]amerge=inputs=2[a]"
    ),
    "-map", "[v]",
    "-map", "[a]",
    "-t", "120",
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "23",
    "-c:a", "aac",
    "-b:a", "192k",
    "-ac", "2",
    "-shortest",
    str(output_path)
    ]

    # cmd = [
    #     "ffmpeg",
    #     "-y",
    #     "-i", str(left_video),
    #     "-i", str(right_video),
    #     "-i", str(left_audio),
    #     "-i", str(right_audio),
    #     "-filter_complex",
    #     (
    #         "[0:v]fps=30,setsar=1,scale=-2:720[v0];"
    #         "[1:v]fps=30,setsar=1,scale=-2:720[v1];"
    #         "[v0][v1]hstack=inputs=2[v];"
    #         "[2:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,volume=1.0[a0];"
    #         "[3:a]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,volume=1.0[a1];"
    #         "[a0][a1]amix=inputs=2:duration=longest:dropout_transition=0:weights='1 1'[a]"
    #     ),
    #     "-map", "[v]",
    #     "-map", "[a]",
    #     "-t", "120",
    #     "-c:v", "libx264",
    #     "-preset", "veryfast",
    #     "-crf", "23",
    #     "-c:a", "aac",
    #     "-b:a", "192k",
    #     "-ac", "2",
    #     "-shortest",
    #     str(output_path)
    # ]

    

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        failed += 1
        print("FFMPEG FAILED")
        print(result.stderr)
    else:
        created += 1
        print("Saved:", output_path)

print("\n=== DONE ===")
print("Created:", created)
print("Failed :", failed)
print("Skipped:", skipped)

Valid normal conversations: 30

Creating normal interaction for V00_S2052_I00000636
Left video  : data/V00_S2052_I00000636/P1309A/V00_S2052_I00000636_P1309A.mp4
Left audio  : data/V00_S2052_I00000636/P1309A/V00_S2052_I00000636_P1309A.wav
Right video : data/V00_S2052_I00000636/P1307A/V00_S2052_I00000636_P1307A.mp4
Right audio : data/V00_S2052_I00000636/P1307A/V00_S2052_I00000636_P1307A.wav
Saved: dataset/normal_data_amerged/V00_S2052_I00000636__NORMAL__P1309A__P1307A.mp4

Creating normal interaction for V03_S0148_I00000135
Left video  : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.mp4
Left audio  : data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.wav
Right video : data/V03_S0148_I00000135/P1274/V03_S0148_I00000135_P1274.mp4
Right audio : data/V03_S0148_I00000135/P1274/V03_S0148_I00000135_P1274.wav
Saved: dataset/normal_data_amerged/V03_S0148_I00000135__NORMAL__P1319__P1274.mp4

Creating normal interaction for V01_S0223_I00000125
Left video  : data/V01_S0223_I00000125

## 7. Create `dataset.zip`

The derived split-screen representation is archived as:

```text
dataset.zip
```

### Downstream use

`dataset.zip` is consumed by:

```text
02_direct_baseline/
└── 01_direct_monolithic_video_only_baseline.ipynb
```

That notebook uses the NORMAL and Wrong Partner split-concatenated videos generated above.


In [35]:
import shutil

shutil.make_archive("dataset", "zip", "dataset")

'/home/jovyan/conversational-anomaly-mlm/notebooks/dataset.zip'

## 8. Create `data.zip`

The original participant-centric `data/` hierarchy is also archived as:

```text
data.zip
```

### Downstream use

`data.zip` is consumed by:

```text
03_isolated_branches/
└── 04_participation_branch_development.ipynb
```

Unlike `dataset.zip`, this archive preserves the participants as separate audiovisual streams with their associated JSON metadata.


In [1]:
import shutil

shutil.make_archive("data", "zip", "data")

'/home/jovyan/conversational-anomaly-mlm/notebooks/data.zip'

## Outputs and relationship between the two archives

| Archive | Representation | Main downstream notebook |
|---|---|---|
| `data.zip` | Separate participant-centric MP4/WAV/JSON records | `03_isolated_branches/04_participation_branch_development.ipynb` |
| `dataset.zip` | Derived split-screen NORMAL and Wrong Partner dyadic videos with merged stereo audio | `02_direct_baseline/01_direct_monolithic_video_only_baseline.ipynb` |

The duplication is historical rather than methodological: both archives originate from the same early source material and serve the same overall data-preparation stage, but different representations were persisted to simplify the first experiments while the thesis pipeline was still being developed.

Later stages of the dissertation use a more systematic and unified controlled-data protocol; those stages are documented in the subsequent notebooks.
